In [ ]:
# 1. Instalasi Dependensi
!pip install diffusers transformers accelerate fastapi uvicorn pyngrok nest-asyncio

import os
import torch
from google.colab import userdata
from diffusers import StableDiffusionXLPipeline
from fastapi import FastAPI
from pydantic import BaseModel
from pyngrok import ngrok
import nest_asyncio
import uvicorn
import base64
from io import BytesIO
import threading
import uvicorn

# 2. Autentikasi menggunakan Token dari Colab Secrets
hf_token = userdata.get('HF_TOKEN')
ngrok_token = userdata.get('NGROK_TOKEN')

# Autentikasi ngrok
ngrok.set_auth_token(ngrok_token)

# 3. Memuat Model Stable Diffusion XL ke GPU (Membutuhkan waktu untuk mengunduh ~7.11 GB)
print("Memuat model ke GPU...")
pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
    use_safetensors=True,
    token=hf_token
)
pipe = pipe.to("cuda")
print("Model berhasil dimuat!")

# 4. Membuat REST API dengan FastAPI
app = FastAPI()

# Skema request data
class ImageRequest(BaseModel):
    prompt: str
    guidance_scale: float = 7.5

@app.post("/generate")
def generate_image(req: ImageRequest):
    # Proses inferensi AI
    image = pipe(req.prompt, guidance_scale=req.guidance_scale).images[0]

    # Mengonversi gambar menjadi string Base64
    buffered = BytesIO()
    image.save(buffered, format="PNG")
    img_str = base64.b64encode(buffered.getvalue()).decode("utf-8")

    return {"image_base64": img_str}

# 5. Menjalankan Server dan Mengekspos URL Publik via ngrok
ngrok_tunnel = ngrok.connect(8000)
print(f"\n=======================================================")
print(f"API PUBLIK ANDA BERJALAN DI: {ngrok_tunnel.public_url}/generate")
print(f"=======================================================\n")

# Menjalankan Uvicorn server secara asinkron di Colab
# 1. Buat fungsi khusus untuk menjalankan server Uvicorn
def run_server():
    # log_level="error" ditambahkan agar output Colab kamu tidak penuh dengan log Uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="error")

# 2. Jalankan fungsi tersebut di dalam sebuah thread yang terpisah
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

print("Server FastAPI berhasil berjalan di background!")

Memuat model ke GPU...


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Model berhasil dimuat!

API PUBLIK ANDA BERJALAN DI: https://7505-35-199-188-75.ngrok-free.app/generate

Server FastAPI berhasil berjalan di background!


In [ ]:
!pip install gradio

In [ ]:
import gradio as gr

# Fungsi ini akan dipanggil saat tombol "Generate" ditekan di UI Gradio
def generate_image_ui(prompt, guidance_scale):
    # Menggunakan model 'pipe' yang sudah dimuat sebelumnya di VRAM GPU
    image = pipe(prompt, guidance_scale=guidance_scale).images[0]
    return image

# Membangun antarmuka web Gradio
interface = gr.Interface(
    fn=generate_image_ui,
    inputs=[
        gr.Textbox(label="Masukkan Prompt Teks", placeholder="Contoh: A futuristic city at sunset..."),
        gr.Slider(minimum=1.0, maximum=20.0, value=7.5, step=0.5, label="Guidance Scale")
    ],
    outputs=gr.Image(label="Hasil Gambar"),
    title="Stable Diffusion XL - Image Generator",
    description="Generator gambar AI gratis yang berjalan di GPU Google Colab."
)

# Menjalankan Gradio dan membuat URL publik
interface.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://00bfe8c72b2e5a302e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


  0%|          | 0/50 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/diffusers/pipelines/stable_diffusion_xl/pipeline_stable_diffusion_xl.py:748: FutureWarning: `upcast_vae` is deprecated and will be removed in version 1.0.0. `upcast_vae` is deprecated. Please use `pipe.vae.to(torch.float32)`. For more details, please refer to: https://github.com/huggingface/diffusers/pull/12619#issue-3606633695.
  deprecate(


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://00bfe8c72b2e5a302e.gradio.live
